<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [17]:
# Building the modeling dataset

model_df = df.copy()

# Binary target: 1 = declining, 0 = not declining
model_df["is_declining"] = (
    model_df["trend_direction"] == "down"
).astype(int)

print("Modeling dataset shape:", model_df.shape)

print("\nTarget distribution:")
print(model_df["is_declining"].value_counts())

print("\nTarget proportions:")
print(model_df["is_declining"].value_counts(normalize=True))

Modeling dataset shape: (30000, 45)

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
is_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [18]:
# Defining model features
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "search_volume",
    "competition",
    "cpc"
]

X = model_df[feature_cols].copy()
y = model_df["is_declining"].copy()
groups = model_df["client_id"].copy()

print("Features:")
print(feature_cols)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nForbidden columns present:")
forbidden = ["trend_direction", "trend_pct", "is_declining_label", "is_declining"]
print([c for c in forbidden if c in X.columns])

Features:
['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 'word_count', 'search_volume', 'competition', 'cpc']

X shape: (30000, 8)
y shape: (30000,)

Forbidden columns present:
[]


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression**

I will use Logistic Regression as the first modeling approach because this lane is a binary prioritization problem and the model provides a simple, interpretable baseline for comparison with the Week-4 rule-based score.

Logistic Regression is appropriate because it estimates the probability of the target outcome from the available features and provides coefficients that can be inspected to understand which signals influence the prediction.

I prefer this model before trying more complex models because the goal is to establish whether a statistical model provides a meaningful improvement over the Week-4 baseline, rather than adding complexity without evidence of better performance.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

The model will be evaluated using a client-grouped split so that content from the same client does not appear in both the training and evaluation sets. This reduces the risk that the model learns client-specific patterns rather than generalizable relationships.

The evaluation will use the same target definition and comparable evaluation data as the Week-4 baseline so that the model and baseline can be compared fairly.

In [20]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Clients in both train and test:",
    len(set(groups_train) & set(groups_test))
)

Train rows: 23837
Test rows: 6163
Clients in both train and test: 0


In [21]:
print("Training missing values:")
print(X_train.isna().sum())

print("\nTest missing values:")
print(X_test.isna().sum())

Training missing values:
days_since_last_update       0
impressions_90d              0
ctr                          0
avg_position                 0
word_count                6614
search_volume             2319
competition               2319
cpc                       2319
dtype: int64

Test missing values:
days_since_last_update       0
impressions_90d              0
ctr                          0
avg_position                 0
word_count                1085
search_volume              149
competition                149
cpc                        149
dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print(model)

model.fit(X_train, y_train)

print("Model trained successfully.")

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])
Model trained successfully.


In [23]:
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions on the held-out test set
y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[ 480 2534]
 [ 461 2688]]

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.16      0.24      3014
           1       0.51      0.85      0.64      3149

    accuracy                           0.51      6163
   macro avg       0.51      0.51      0.44      6163
weighted avg       0.51      0.51      0.45      6163



In [24]:
from sklearn.metrics import precision_score
import numpy as np

# Model predictions and probabilities
model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

# Recreate the EXACT W04 baseline rule on the W05 test set

baseline_test = model_df.iloc[test_idx].copy()

# Staleness score
baseline_test["staleness_score"] = np.select(
    [
        baseline_test["days_since_last_update"].between(0, 90),
        baseline_test["days_since_last_update"].between(91, 180),
        baseline_test["days_since_last_update"] >= 181
    ],
    [0, 1, 2],
    default=0
)

# Visibility score
baseline_test["visibility_score"] = np.select(
    [
        baseline_test["impressions_90d"].between(0, 100),
        baseline_test["impressions_90d"].between(101, 10000),
        baseline_test["impressions_90d"] >= 10001
    ],
    [0, 1, 2],
    default=0
)

# EXACT W04 score
baseline_test["baseline_score"] = (
    baseline_test["staleness_score"]
    + baseline_test["visibility_score"]
)

# Same ranking logic as W04
baseline_ranked = baseline_test.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
)

# Logistic Regression ranking
model_ranked = model_df.iloc[test_idx].copy()
model_ranked["model_probability"] = model_prob

model_ranked = model_ranked.sort_values(
    "model_probability",
    ascending=False
)

# Precision@10
k = 10

baseline_precision_10 = (
    baseline_ranked.head(k)["is_declining"].mean()
)

model_precision_10 = (
    model_ranked.head(k)["is_declining"].mean()
)

comparison = pd.DataFrame({
    "approach": [
        "W04 baseline",
        "Logistic Regression"
    ],
    "precision_at_10": [
        baseline_precision_10,
        model_precision_10
    ]
})

print("Model-vs-baseline comparison:")
display(comparison)

Model-vs-baseline comparison:


,approach,precision_at_10
0,W04 baseline,0.3
1,Logistic Regression,0.4


In [25]:
# Add the overall test-set base rate
base_rate = y_test.mean()

comparison_with_base = pd.DataFrame({
    "approach": [
        "Base rate",
        "W04 baseline",
        "Logistic Regression"
    ],
    "precision_at_10": [
        base_rate,
        baseline_precision_10,
        model_precision_10
    ]
})

print("Final model-vs-baseline comparison:")
display(comparison_with_base)

Final model-vs-baseline comparison:


,approach,precision_at_10
0,Base rate,0.510952
1,W04 baseline,0.300000
2,Logistic Regression,0.400000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Logistic Regression model improved Precision@10 from 0.30 for the Week-4 baseline to 0.40, indicating that the model identified more declining-content items among its highest-ranked recommendations.

The errors show that the model's ranking is not perfect: some highly ranked items are not declining, while some declining items may receive lower probabilities. This is expected because the model uses only eight current-state features and does not capture every factor that can influence content decline.

The result suggests that the learned feature relationships provide some additional signal beyond the hand-written baseline, but the improvement is modest and should not be interpreted as proof that the model will generalize to every client or future dataset.

In [26]:
# Identify concrete false-positive and false-negative cases

error_df = model_df.iloc[test_idx].copy()

error_df["predicted"] = model_pred
error_df["probability"] = model_prob

error_df["error_type"] = np.select(
    [
        (error_df["is_declining"] == 0) & (error_df["predicted"] == 1),
        (error_df["is_declining"] == 1) & (error_df["predicted"] == 0)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

errors = error_df[
    error_df["error_type"] != "Correct"
].copy()

print("Total errors:", len(errors))

print("\nFalse positives:")
display(
    errors[errors["error_type"] == "False Positive"][
        [
            "client_id",
            "content_id",
            "is_declining",
            "predicted",
            "probability",
            "days_since_last_update",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ].head(3)
)

print("\nFalse negatives:")
display(
    errors[errors["error_type"] == "False Negative"][
        [
            "client_id",
            "content_id",
            "is_declining",
            "predicted",
            "probability",
            "days_since_last_update",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ].head(3)
)

Total errors: 2995

False positives:


,client_id,content_id,is_declining,predicted,probability,days_since_last_update,impressions_90d,ctr,avg_position
13,client_8527a891e2,content_a5a2fbc76336,0,1,0.549230,103,307,0.00,39.8
36,client_f369cb89fc,content_bce275871a25,0,1,0.511179,20,371,1.35,5.4
56,client_f369cb89fc,content_dcebfd222b10,0,1,0.546550,20,16,0.00,4.6



False negatives:


,client_id,content_id,is_declining,predicted,probability,days_since_last_update,impressions_90d,ctr,avg_position
1,client_4e07408562,content_a1fb4e703a9e,1,0,0.496799,25,15320,0.05,20.3
49,client_8527a891e2,content_f0717373e86e,1,0,0.488501,8,9,0.00,10.1
60,client_f369cb89fc,content_b9104a222d01,1,0,0.489427,20,25,4.00,6.2


### Three concrete wrong cases

1. **False positive — content_a5a2fbc76336:** The model predicted decline with probability 0.549, but the observed label was non-declining. The page was relatively stale at 103 days, which likely pushed the model toward a decline prediction, even though its current search visibility was only 307 impressions and CTR was recorded as 0.00 in the dataset.

2. **False negative — content_a1fb4e703a9e:** The model predicted non-decline with probability 0.497, but the observed label was declining. The page had only 25 days of staleness but very high visibility (15,320 impressions), showing that high current visibility does not guarantee that the content will not decline.

3. **False negative — content_f0717373e86e:** The model predicted non-decline with probability 0.489, while the observed label was declining. The page was recently updated (8 days) and had only 9 impressions, illustrating that the model can miss declining content when the available current-state signals do not strongly indicate decline.

In [27]:
# Inspect Logistic Regression feature coefficients

coefficients = model.named_steps["classifier"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": coefficients
})

feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
)

display(feature_importance)

,feature,coefficient,abs_coefficient
2,ctr,-0.197160,0.197160
0,days_since_last_update,0.189355,0.189355
4,word_count,0.132958,0.132958
1,impressions_90d,-0.081225,0.081225
3,avg_position,-0.075389,0.075389
5,search_volume,-0.020335,0.020335
6,competition,0.020037,0.020037
7,cpc,-0.015943,0.015943


### What the model leans on

The three strongest coefficients by absolute magnitude are CTR, days since last update, and word count. CTR has a negative coefficient, meaning higher CTR is associated with a lower predicted probability of decline. Days since last update and word count have positive coefficients, meaning higher values are associated with a higher predicted probability of decline.

These relationships are plausible for a content-decline task, but they should be interpreted as model associations rather than causal effects.

### Feature interpretation

The model relies most strongly on CTR and days since last update. The negative CTR coefficient indicates that higher CTR is associated with a lower predicted probability of decline, while the positive coefficient for days since last update indicates that greater staleness is associated with a higher predicted probability of decline.

Word count has the next-largest positive coefficient, while impressions and average position have smaller negative coefficients. The remaining features have relatively small coefficients, suggesting weaker influence in this Logistic Regression model.

In [29]:
print("W05 SELF-CHECK")
print("=" * 40)

print("\n1. Target leakage check:")
forbidden = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "is_declining"
]
print("Forbidden features present:",
      [c for c in feature_cols if c in forbidden])

print("\n2. Group split check:")
print(
    "Clients in both train and test:",
    len(set(groups_train) & set(groups_test))
)

print("\n3. Feature count:")
print("Number of model features:", len(feature_cols))

print("\n4. Baseline vs model:")
display(comparison_with_base)

print("\n5. Model interpretation:")
print("Top 3 features by absolute coefficient:")
display(feature_importance.head(3))

print("\n6. Reproducibility:")
print("Random state: 42")

W05 SELF-CHECK

1. Target leakage check:
Forbidden features present: []

2. Group split check:
Clients in both train and test: 0

3. Feature count:
Number of model features: 8

4. Baseline vs model:


,approach,precision_at_10
0,Base rate,0.510952
1,W04 baseline,0.300000
2,Logistic Regression,0.400000



5. Model interpretation:
Top 3 features by absolute coefficient:


,feature,coefficient,abs_coefficient
2,ctr,-0.197160,0.197160
0,days_since_last_update,0.189355,0.189355
4,word_count,0.132958,0.132958



6. Reproducibility:
Random state: 42


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.